# Django Class-based Views

En este ejercicio se creó una vista basada en clases llamada ProtectedListView.

La vista utiliza LoginRequiredMixin para requerir autenticación y sobrescribe
get_queryset() para mostrar únicamente los productos pertenecientes al usuario
que realiza la petición.


## Modelo ProductModel

In [ ]:
from django.conf import settings
from django.db import models


class ProductModel(models.Model):
    name = models.CharField(max_length=100)
    price = models.DecimalField(max_digits=10, decimal_places=2)
    description = models.TextField()
    seller = models.CharField(max_length=100)
    color = models.CharField(max_length=50)
    product_dimensions = models.CharField(max_length=100)

    user = models.ForeignKey(
        settings.AUTH_USER_MODEL,
        on_delete=models.CASCADE,
        related_name="products",
        null=True,
        blank=True,
    )

    def __str__(self):
        return self.name


## ProtectedListView

In [ ]:
from django.contrib.auth.mixins import LoginRequiredMixin
from django.views.generic import ListView

from .models import ProductModel


class ProtectedListView(LoginRequiredMixin, ListView):
    model = ProductModel
    template_name = "ejercicios/product_list.html"
    context_object_name = "products"

    def get_queryset(self):
        return ProductModel.objects.filter(user=self.request.user)


## Endpoint my-products

In [ ]:
from django.urls import path
from ejercicios.views import ProtectedListView


urlpatterns = [
    path(
        "my-products/",
        ProtectedListView.as_view(),
        name="my-products"
    ),
]


## Pruebas realizadas

1. Se creó un usuario de prueba llamado Mark.
2. Se asignó Producto 1 al usuario Mark.
3. Se comprobó que Mark tiene solamente un producto asociado.
4. Al acceder a /my-products/ sin iniciar sesión, Django redirigió a:
   /accounts/login/?next=/my-products/
5. Se inició sesión como Mark mediante el administrador de Django.
6. Al acceder nuevamente a /my-products/, solamente apareció Producto 1.

Por lo tanto, LoginRequiredMixin protege correctamente la vista y
get_queryset() filtra los productos de acuerdo con request.user.
